# JetRacer Lane Fusion — Live TensorRT UI

ResNet18 dự đoán waypoint `(x, y, confidence)`, sau đó EMA → state machine → PD → low-pass/rate limiter → adaptive throttle. Mặc định **DISARMED**; hãy kê bánh xe và kiểm tra chiều lái trước khi bật motor. Chạy các cell từ trên xuống.

In [1]:
import sys, time, csv, threading
from pathlib import Path
import cv2, numpy as np, ipywidgets as widgets
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'waypoint_lane_fusion' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg
from notebook3.basic_motion import JetRacerController
from waypoint_lane_fusion.config import load_config, resolve_path
from waypoint_lane_fusion.lane_model import OnnxWaypointModel, TensorRTWaypointModel
from waypoint_lane_fusion.controller import WaypointFilter, DriveController
from waypoint_lane_fusion.behavior import BehaviorStateMachine
from waypoint_lane_fusion.telemetry import overlay
from waypoint_lane_fusion.types import DetectionSnapshot

cfg = load_config(PROJECT_ROOT / 'waypoint_lane_fusion/config.yaml')
engine = PROJECT_ROOT / 'waypoint_lane_fusion/artifacts/lane_resnet18_bootstrap_fp16.engine'
onnx = PROJECT_ROOT / 'waypoint_lane_fusion/artifacts/lane_resnet18_bootstrap.onnx'
lane_model = TensorRTWaypointModel(engine) if engine.exists() else OnnxWaypointModel(onnx)
backend_name = 'TensorRT FP16' if engine.exists() else 'ONNX fallback'
print('Lane backend:', backend_name)


WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


Lane backend: TensorRT FP16


Nếu camera đang bị khóa, chạy `sudo systemctl restart nvargus-daemon` trong terminal trước cell tiếp theo. Notebook không nhúng mật khẩu sudo.

In [2]:
try:
    camera.running = False; camera.unobserve_all()
except Exception:
    pass
camera = CSICamera(width=224, height=224, capture_fps=0)
car = JetRacerController(cfg['hardware']['steering_gain'], cfg['hardware']['steering_offset'], cfg['hardware']['throttle_gain'], cfg['control']['throttle_max'])
car.stop(); car.center_steering()
waypoint_filter = WaypointFilter(cfg['control']['waypoint_ema'])
behavior = BehaviorStateMachine(cfg['control'])
controller = DriveController(cfg['control'])
snapshot = DetectionSnapshot()


[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!


In [3]:
state = widgets.ToggleButtons(options=['stop','live'], value='stop', description='State')
armed = widgets.Checkbox(value=False, description='ARM MOTOR')
max_throttle = widgets.FloatSlider(value=cfg['control']['throttle_max'], min=0.0, max=0.5, step=0.005, description='Max throttle')
steering_scale = widgets.FloatSlider(value=1.0, min=0.5, max=2, step=0.05, description='Steer scale')
raw_view = widgets.Image(format='jpeg', width=224, height=224)
debug_view = widgets.Image(format='jpeg', width=224, height=224)
status = widgets.HTML(value='<b>STOPPED / DISARMED</b>')
blank = np.zeros((224,224,3), np.uint8); raw_view.value=bgr8_to_jpeg(blank); debug_view.value=bgr8_to_jpeg(blank)
display(widgets.VBox([widgets.HBox([raw_view,debug_view]), status, widgets.HBox([state,armed]), max_throttle, steering_scale]))


In [4]:
callback_lock = threading.Lock(); last_tick = time.perf_counter(); fps_ema = 0.0
log_dir = PROJECT_ROOT / 'waypoint_lane_fusion/logs'; log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / time.strftime('live_%Y%m%d_%H%M%S.csv')
log_stream = log_path.open('w', newline=''); log_writer = csv.writer(log_stream); log_buffer=[]
log_writer.writerow(['timestamp','fps','x','y','confidence','steering_raw','steering','throttle','state','armed'])

def live_update(change):
    global last_tick, fps_ema
    if state.value != 'live' or not callback_lock.acquire(False): return
    try:
        frame = change['new']; now = time.perf_counter(); dt=max(0.005,now-last_tick); last_tick=now
        raw = lane_model.predict(frame); filtered = waypoint_filter.update(raw)
        drive_state,bias = behavior.update(filtered,snapshot)
        command = controller.update(filtered,drive_state,dt,bias)
        command.steering = float(np.clip(command.steering*steering_scale.value,-1,1))
        command.throttle = min(command.throttle,max_throttle.value)
        if armed.value and drive_state.value not in ('LANE_LOST','OBSTACLE','RED_LIGHT'):
            car.set_steering(command.steering); car.set_throttle(command.throttle)
        else:
            car.stop(); car.center_steering()
        instant=1.0/dt; fps_ema=instant if fps_ema==0 else .2*instant+.8*fps_ema
        rendered=overlay(frame,raw,filtered,command,snapshot,fps_ema)
        raw_view.value=bgr8_to_jpeg(frame); debug_view.value=bgr8_to_jpeg(rendered)
        status.value='<b>%s | %s | FPS %.1f | conf %.2f | steer %+.3f | throttle %.3f</b>' % (backend_name,drive_state.value,fps_ema,filtered.confidence,command.steering,command.throttle)
        log_buffer.append([time.time(),fps_ema,raw.x,raw.y,filtered.confidence,command.steering_raw,command.steering,command.throttle,drive_state.value,int(armed.value)])
        if len(log_buffer) >= 20: log_writer.writerows(log_buffer); log_stream.flush(); log_buffer.clear()
    except Exception as exc:
        car.stop(); car.center_steering(); state.value='stop'; status.value='<b style="color:red">ERROR: %s</b>' % exc
    finally:
        callback_lock.release()

def safety_changed(change):
    if change.get('new') == 'stop' or not armed.value:
        car.stop(); car.center_steering()
        if change.get('new') == 'stop' and log_buffer: log_writer.writerows(log_buffer); log_stream.flush(); log_buffer.clear()
state.observe(safety_changed,names='value'); armed.observe(safety_changed,names='value')
camera.observe(live_update,names='value'); camera.running=True
print('Camera running. Chọn live để infer; ARM MOTOR chỉ sau khi kê bánh.')


Camera running. Chọn live để infer; ARM MOTOR chỉ sau khi kê bánh.


## Dừng an toàn — luôn chạy cell này trước khi đóng notebook

In [ ]:
state.value='stop'; armed.value=False; car.stop(); car.center_steering()
camera.running=False; camera.unobserve_all()
if log_buffer: log_writer.writerows(log_buffer); log_buffer.clear()
log_stream.flush(); log_stream.close()
print('Stopped safely. Log:', log_path)
